# Phase 1

In [ ]:
status: results  of initial run

In [6]:
#  Install libraries
# your choice resolve conflicts in current or create new environ
#!pip uninstall -y sparkmagic
!pip install -r require_phase1_20250316.txt -q

In [7]:
#  Import libraries
import dspy
import ipywidgets as widgets
from IPython.display import display
import boto3
import PyPDF2
import json
import os
import faiss
import io  # Import io
import numpy as np # Import numpy
from typing import List, Dict
from datetime import datetime
import re
import unicodedata

In [27]:
# Define Configuration Dictionary
#  import config #prod version

CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "pdf_folder": "dev/pdf/arxiv_wellsfargo",
    "pages_folder": "dev/page",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "embedding_model": "amazon.titan-embed-text-v1",  # Bedrock model ID
    "embedding_dimension": 1536,  # Specify the dimension for the Bedrock model you are using
    "chunk_size": 200,
    "nlist": 512,  # Number of Voronoi cells for IndexIVFFlat
    "nprobe": 16   # Number of Voronoi cells to search during query
}

In [20]:
# Initialize S3 client
s3 = boto3.client('s3')

# Configuration Widgets
bucket_name_widget = widgets.Text(description="Bucket Name:", placeholder=CONFIG["bucket_name"])
pdf_folder_widget = widgets.Text(description="PDF Folder:", placeholder=CONFIG["pdf_folder"])
pages_folder_widget = widgets.Text(description="Pages Folder:", placeholder=CONFIG["pages_folder"])
json_folder_widget = widgets.Text(description="JSON Folder:", placeholder=CONFIG["json_folder"])
index_folder_widget = widgets.Text(description="Index Folder:", placeholder=CONFIG["index_folder"])
chunk_size_widget = widgets.IntText(description="Chunk Size:", value=CONFIG["chunk_size"])

# Set Default Display Configuration Widget Values
display(bucket_name_widget, pdf_folder_widget, pages_folder_widget, json_folder_widget,
        chunk_size_widget)

Text(value='', description='Bucket Name:', placeholder='sagemaker-us-east-1-188494237500')

Text(value='', description='PDF Folder:', placeholder='dev/pdf/arxiv_wellsfargo')

Text(value='', description='Pages Folder:', placeholder='dev/page')

Text(value='', description='JSON Folder:', placeholder='dev/json')

IntText(value=200, description='Chunk Size:')

In [23]:
def tag_s3_object(bucket: str, key: str, tag_key: str, tag_value: str) -> None:
    """Adds or updates a tag on an S3 object."""
    try:
        s3.put_object_tagging(
            Bucket=bucket,
            Key=key,
            Tagging={'TagSet': [{'Key': tag_key, 'Value': tag_value}]}
        )
        print(f"Successfully tagged object {key} in bucket {bucket} with {tag_key}:{tag_value}")
    except Exception as e:
        print(f"Error tagging object {key}: {e}")


def check_s3_object_tag(bucket: str, key: str, tag_key: str) -> bool:
    """Checks if an S3 object has a specific tag."""
    try:
        response = s3.get_object_tagging(Bucket=bucket, Key=key)
        tag_set = response.get('TagSet', [])
        for tag in tag_set:
            if tag['Key'] == tag_key:
                return True
        return False
    except Exception as e:
        # Handle exceptions, like object not found or no tagging permissions.
        # Depending on your needs, you might want to log this or raise an exception.
        print(f"Error checking tag for object {key}: {e}")
        return False  # Assume no tag if there's an error


def clean_text(text):
    """
    Cleans extracted text by applying common text preprocessing steps.
    - Fixes common OCR artifacts
    - Standardizes formatting
    - Removes unwanted characters and excessive newlines
    """

    # Normalize Unicode characters (e.g., convert fancy quotes to ASCII)
    text = unicodedata.normalize("NFKD", text)

    # Remove common PDF artifacts (e.g., incorrect range values)
    text = text.replace('0.00-10', '0.00')

    # Fix hyphenated line breaks (common in PDF text extraction)
    text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text)

    # Remove multiple spaces, newlines, and excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove multiple newlines but **keep paragraph breaks**
    text = re.sub(r'\n{2,}', '\n', text)  # Keeps a single \n for meaningful breaks

    # Optional: Remove all `\n` if not needed at all
    text = text.replace("\n", " ")

    # Remove special characters that don’t contribute to meaning
    text = re.sub(r'[^\w\s.,;!?%$-]', '', text)  # Keeps punctuation but removes junk

    return text


def split_pdf_into_pages_s3(config: Dict, pdf_key: str) -> List[str]:
    """Splits a PDF in S3 into individual page PDF files."""

    try:
        # Download PDF
        response = s3.get_object(Bucket=config["bucket_name"], Key=pdf_key)
        pdf_file = response['Body'].read()

        # Create PDF reader object
        pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_file))

        num_pages = len(pdf_reader.pages)
        output_keys = []
        # Iterate over each page
        for page_num in range(num_pages):
            page = pdf_reader.pages[page_num]

            # Create a new PDF writer object for each page
            pdf_writer = PyPDF2.PdfWriter()
            pdf_writer.add_page(page)

            # Save the single page PDF to memory
            with io.BytesIO() as output_buffer:
                pdf_writer.write(output_buffer)
                output_pdf_file = output_buffer.getvalue()

            # Upload the single page PDF to S3
            page_key = f"{config['pages_folder']}/page_{page_num + 1}.pdf"
            s3.put_object(Bucket=config["bucket_name"], Key=page_key, Body=output_pdf_file)
            output_keys.append(page_key)

        return output_keys

    except Exception as e:
        print(f"Error splitting PDF: {e}")
        return []


def extract_text_from_pdf_page_s3(config: Dict, pdf_page_key: str) -> Dict:
    """Extracts text and metadata from a single-page PDF in S3."""
    try:
        # Download PDF
        response = s3.get_object(Bucket=config["bucket_name"], Key=pdf_page_key)
        pdf_file = response['Body'].read()

        pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_file))
        page = pdf_reader.pages[0]  # Assuming single-page PDF

        text = page.extract_text()
        metadata = pdf_reader.metadata  # might be None

        data = {
            "text": text,
            "metadata": str(metadata),  # Convert to string for JSON compatibility
            "page_number": int(pdf_page_key.split("_")[-1].split(".")[0]) #example: page_1.pdf, we would extract "1".
        }

        return data  # Returns a Python dictionary
    except Exception as e:
        print(f"Error extracting data from PDF: {e}")
        return {}


def chunk_text_into_segments(pdf_page_data: Dict, config:Dict) -> List[str]:
    """Chunks text from PDF page data into fixed-size segments."""
    try:
        text = pdf_page_data['text']

        # Clean the text before chunking
        cleaned_text = clean_text(text)

        words = cleaned_text.split()
        chunks = []
        for i in range(0, len(words), config["chunk_size"]):
            chunk = " ".join(words[i:i + config["chunk_size"]])
            chunks.append(chunk)

        return chunks
    except Exception as e:
        print(f"Error chunking text: {e}")
        return []


def save_text_chunk_as_json_s3(config: Dict, original_key:str, chunks: List[str]) -> List[str]:
    """Creates JSON files for each text chunk and stores them in S3."""
    output_keys = []
    try:
        page_number = original_key.split("_")[-1].split(".")[0]
        for i, chunk in enumerate(chunks):
            chunk_data = {"text": chunk, "page_number": page_number, "chunk_number": i + 1}
            chunk_key = f"{config['json_folder']}/page_{page_number}_chunk_{i+1}.json"
            s3.put_object(Bucket=config["bucket_name"], Key=chunk_key, Body=json.dumps(chunk_data).encode('utf-8'))
            output_keys.append(chunk_key)

        # Tag the original PDF page as processed
        tag_s3_object(config["bucket_name"], original_key, "doc_processed", datetime.now().isoformat())

        return output_keys
    except Exception as e:
        print(f"Error creating JSON from chunks: {e}")
        return []

def list_objects_s3(bucket: str, prefix: str, tag_key: str = "doc_processed") -> List[str]:
    """Lists all objects in an S3 bucket with a prefix that do NOT have a specific tag."""
    try:
        response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
        objects = response.get('Contents', [])
        object_keys = []

        for obj in objects:
            key = obj['Key']
            if not check_s3_object_tag(bucket, key, tag_key):
                object_keys.append(key)

        return object_keys
    except Exception as e:
        print(f"Error listing objects in S3: {e}")
        return []

def process_documents(button):
    # Update the config
    CONFIG["bucket_name"] = bucket_name_widget.value
    CONFIG["pdf_folder"] = pdf_folder_widget.value
    CONFIG["pages_folder"] = pages_folder_widget.value
    CONFIG["json_folder"] = json_folder_widget.value
    CONFIG["chunk_size"] = int(chunk_size_widget.value)

    # Get list of PDFs from S3 that have NOT been processed
    pdf_keys = list_objects_s3(CONFIG["bucket_name"], CONFIG["pdf_folder"])
    print(f"Found {len(pdf_keys)} PDFs to process.")

    # Process each PDF
    for pdf_key in pdf_keys:
        print(f"Processing PDF: {pdf_key}")

        # 1. Split the PDF into pages
        page_keys = split_pdf_into_pages_s3(CONFIG, pdf_key)

        # Process each page
        for page_key in page_keys:

            # 2. Extract text from the PDF page
            pdf_data = extract_text_from_pdf_page_s3(CONFIG, page_key)

            # 3. Chunk the extracted text
            chunks = chunk_text_into_segments(pdf_data, CONFIG)

            # 4. Save the chunks as JSON in S3
            chunk_keys = save_text_chunk_as_json_s3(CONFIG, page_key, chunks)

    print(f"Completed whole process.")


In [24]:
#test

In [22]:
process_button = widgets.Button(description="Process Documents")
process_button.on_click(process_documents)
display(process_button)


Button(description='Process Documents', style=ButtonStyle())

In [28]:
CONFIG

{'bucket_name': 'sagemaker-us-east-1-188494237500',
 'pdf_folder': 'dev/pdf/arxiv_wellsfargo',
 'pages_folder': 'dev/page',
 'json_folder': 'dev/json',
 'index_folder': 'dev/idx',
 'embedding_model': 'amazon.titan-embed-text-v1',
 'embedding_dimension': 1536,
 'chunk_size': 200,
 'nlist': 512,
 'nprobe': 16}

In [29]:
import boto3
import io
import os
import json
from typing import List, Dict
from PyPDF2 import PdfReader, PdfWriter
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# AWS S3 client
s3 = boto3.client("s3")

def check_s3_object_tag(bucket: str, key: str, tag_key: str) -> bool:
    """Checks if an object in S3 has a specific tag."""
    try:
        response = s3.get_object_tagging(Bucket=bucket, Key=key)
        tags = {tag["Key"]: tag["Value"] for tag in response.get("TagSet", [])}
        return tag_key in tags
    except s3.exceptions.ClientError as e:
        if e.response['Error']['Code'] == 'AccessDenied':
            logging.warning(f"Skipping tag check for {key} due to insufficient permissions.")
            return False  # Assume the object has no tag and continue
        else:
            logging.error(f"Error retrieving tags for {key}: {e}")
            return False

def list_objects_s3(bucket: str, prefix: str, tag_key: str = "doc_processed") -> List[str]:
    """Lists all objects in an S3 bucket with a prefix that do NOT have a specific tag."""
    try:
        response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
        objects = response.get('Contents', [])
        logging.info(f"Found {len(objects)} objects in {bucket}/{prefix}")

        object_keys = [obj['Key'] for obj in objects if not check_s3_object_tag(bucket, obj['Key'], tag_key)]
        
        logging.info(f"Completed listing objects in {bucket}/{prefix}. {len(object_keys)} files found.")  # ✅ STATUS MESSAGE
        return object_keys
    except Exception as e:
        logging.error(f"Error listing objects in S3: {e}")
        return []

def split_pdf_into_pages_s3(pdf_key: str, config: Dict):
    """Splits a PDF into individual pages and uploads them to S3."""
    try:
        bucket = config["bucket_name"]
        response = s3.get_object(Bucket=bucket, Key=pdf_key)
        pdf_file = response["Body"].read()

        reader = PdfReader(io.BytesIO(pdf_file))
        base_name = os.path.splitext(os.path.basename(pdf_key))[0]

        for page_num, page in enumerate(reader.pages):
            output_pdf = io.BytesIO()
            writer = PdfWriter()
            writer.add_page(page)
            writer.write(output_pdf)
            output_pdf.seek(0)

            page_key = f"{config['pages_folder']}/{base_name}_page_{page_num + 1}.pdf"
            s3.put_object(Bucket=bucket, Key=page_key, Body=output_pdf.getvalue())
            logging.info(f"Uploaded page {page_num + 1} to {page_key}")

        logging.info(f"Completed splitting {pdf_key} into pages and uploading to S3.")  # ✅ STATUS MESSAGE
    except Exception as e:
        logging.error(f"Error splitting PDF {pdf_key}: {e}")

def extract_text_from_pdf_page_s3(page_key: str, config: Dict) -> Dict:
    """Extracts text from a single page PDF stored in S3."""
    try:
        bucket = config["bucket_name"]
        response = s3.get_object(Bucket=bucket, Key=page_key)
        pdf_file = response["Body"].read()

        reader = PdfReader(io.BytesIO(pdf_file))
        text = reader.pages[0].extract_text() or ""
        
        logging.info(f"Completed text extraction for {page_key}. Extracted {len(text)} characters.")  # ✅ STATUS MESSAGE
        return {"page_number": int(page_key.split("_")[-1].split(".")[0]), "text": text.strip()}
    except Exception as e:
        logging.error(f"Error extracting text from {page_key}: {e}")
        return {"page_number": None, "text": ""}

def chunk_text_into_segments(pdf_page_data: Dict, config: Dict) -> List[str]:
    """Chunks text from PDF page data into fixed-size segments."""
    try:
        text = pdf_page_data.get('text', '')
        if not text.strip():
            logging.warning(f"Skipping empty text for page {pdf_page_data.get('page_number', 'Unknown')}")
            return []

        words = text.split()
        chunks = [" ".join(words[i:i + config["chunk_size"]]) for i in range(0, len(words), config["chunk_size"])]

        logging.info(f"Completed text chunking for page {pdf_page_data.get('page_number', 'Unknown')}. {len(chunks)} chunks created.")  # ✅ STATUS MESSAGE
        return chunks
    except Exception as e:
        logging.error(f"Error chunking text: {e}")
        return []

def save_json_to_s3(data: Dict, json_key: str, config: Dict):
    """Saves JSON data to an S3 bucket."""
    try:
        bucket = config["bucket_name"]
        s3.put_object(Bucket=bucket, Key=json_key, Body=json.dumps(data))
        logging.info(f"Successfully saved JSON to S3: {json_key}.")  # ✅ STATUS MESSAGE
    except Exception as e:
        logging.error(f"Error saving JSON to S3: {e}")

def process_documents():
    """Processes all PDFs, extracts pages, text, and saves JSON."""
    try:
        pdf_keys = list_objects_s3(CONFIG["bucket_name"], CONFIG["pdf_folder"])
        logging.info(f"Processing {len(pdf_keys)} PDFs...")

        for pdf_key in pdf_keys:
            logging.info(f"Processing PDF: {pdf_key}")
            split_pdf_into_pages_s3(pdf_key, CONFIG)

            # Process each page
            page_keys = list_objects_s3(CONFIG["bucket_name"], CONFIG["pages_folder"])
            all_chunks = []

            for page_key in page_keys:
                pdf_page_data = extract_text_from_pdf_page_s3(page_key, CONFIG)
                chunks = chunk_text_into_segments(pdf_page_data, CONFIG)
                all_chunks.extend(chunks)

            # Save processed data as JSON
            base_name = os.path.splitext(os.path.basename(pdf_key))[0]
            json_key = f"{CONFIG['json_folder']}/{base_name}.json"
            save_json_to_s3({"chunks": all_chunks}, json_key, CONFIG)

            logging.info(f"Finished processing PDF: {pdf_key}. JSON saved successfully.")  # ✅ STATUS MESSAGE

    except Exception as e:
        logging.error(f"Error processing documents: {e}")

if __name__ == "__main__":
    process_documents()


2025-03-16 08:34:26,817 - INFO - Found 12 objects in sagemaker-us-east-1-188494237500/dev/pdf/arxiv_wellsfargo
2025-03-16 08:34:27,006 - INFO - Completed listing objects in sagemaker-us-east-1-188494237500/dev/pdf/arxiv_wellsfargo. 12 files found.
2025-03-16 08:34:27,007 - INFO - Processing 12 PDFs...
2025-03-16 08:34:27,011 - INFO - Processing PDF: dev/pdf/arxiv_wellsfargo/1806.00663.pdf


Error checking tag for object dev/pdf/arxiv_wellsfargo/1806.00663.pdf: An error occurred (AccessDenied) when calling the GetObjectTagging operation: User: arn:aws:sts::188494237500:assumed-role/AmazonSageMaker-ExecutionRole-20250122T114147/SageMaker is not authorized to perform: s3:GetObjectTagging on resource: "arn:aws:s3:::sagemaker-us-east-1-188494237500/dev/pdf/arxiv_wellsfargo/1806.00663.pdf" because no identity-based policy allows the s3:GetObjectTagging action
Error checking tag for object dev/pdf/arxiv_wellsfargo/1806.01933.pdf: An error occurred (AccessDenied) when calling the GetObjectTagging operation: User: arn:aws:sts::188494237500:assumed-role/AmazonSageMaker-ExecutionRole-20250122T114147/SageMaker is not authorized to perform: s3:GetObjectTagging on resource: "arn:aws:s3:::sagemaker-us-east-1-188494237500/dev/pdf/arxiv_wellsfargo/1806.01933.pdf" because no identity-based policy allows the s3:GetObjectTagging action
Error checking tag for object dev/pdf/arxiv_wellsfargo/2

2025-03-16 08:34:27,159 - INFO - Uploaded page 1 to dev/page/1806.00663_page_1.pdf
2025-03-16 08:34:27,241 - INFO - Uploaded page 2 to dev/page/1806.00663_page_2.pdf
2025-03-16 08:34:27,343 - INFO - Uploaded page 3 to dev/page/1806.00663_page_3.pdf
2025-03-16 08:34:27,448 - INFO - Uploaded page 4 to dev/page/1806.00663_page_4.pdf
2025-03-16 08:34:27,557 - INFO - Uploaded page 5 to dev/page/1806.00663_page_5.pdf
2025-03-16 08:34:27,659 - INFO - Uploaded page 6 to dev/page/1806.00663_page_6.pdf
2025-03-16 08:34:27,732 - INFO - Uploaded page 7 to dev/page/1806.00663_page_7.pdf
2025-03-16 08:34:27,813 - INFO - Uploaded page 8 to dev/page/1806.00663_page_8.pdf
2025-03-16 08:34:27,884 - INFO - Uploaded page 9 to dev/page/1806.00663_page_9.pdf
2025-03-16 08:34:28,019 - INFO - Uploaded page 10 to dev/page/1806.00663_page_10.pdf
2025-03-16 08:34:28,112 - INFO - Uploaded page 11 to dev/page/1806.00663_page_11.pdf
2025-03-16 08:34:28,197 - INFO - Uploaded page 12 to dev/page/1806.00663_page_12.pd

Error checking tag for object dev/page/: An error occurred (AccessDenied) when calling the GetObjectTagging operation: User: arn:aws:sts::188494237500:assumed-role/AmazonSageMaker-ExecutionRole-20250122T114147/SageMaker is not authorized to perform: s3:GetObjectTagging on resource: "arn:aws:s3:::sagemaker-us-east-1-188494237500/dev/page/" because no identity-based policy allows the s3:GetObjectTagging action
Error checking tag for object dev/page/1806.00663_page_1.pdf: An error occurred (AccessDenied) when calling the GetObjectTagging operation: User: arn:aws:sts::188494237500:assumed-role/AmazonSageMaker-ExecutionRole-20250122T114147/SageMaker is not authorized to perform: s3:GetObjectTagging on resource: "arn:aws:s3:::sagemaker-us-east-1-188494237500/dev/page/1806.00663_page_1.pdf" because no identity-based policy allows the s3:GetObjectTagging action
Error checking tag for object dev/page/1806.00663_page_10.pdf: An error occurred (AccessDenied) when calling the GetObjectTagging oper

2025-03-16 08:34:28,770 - INFO - Completed listing objects in sagemaker-us-east-1-188494237500/dev/page. 16 files found.
2025-03-16 08:34:28,787 - ERROR - Error extracting text from dev/page/: Cannot read an empty file
2025-03-16 08:34:28,788 - WARNING - Skipping empty text for page None
2025-03-16 08:34:28,861 - INFO - Completed text extraction for dev/page/1806.00663_page_1.pdf. Extracted 3094 characters.
2025-03-16 08:34:28,862 - INFO - Completed text chunking for page 1. 3 chunks created.
2025-03-16 08:34:28,931 - INFO - Completed text extraction for dev/page/1806.00663_page_10.pdf. Extracted 1605 characters.
2025-03-16 08:34:28,933 - INFO - Completed text chunking for page 10. 2 chunks created.


...

2025-03-16 08:36:55,520 - INFO - Completed text extraction for dev/page/1806.00663_page_1.pdf. Extracted 3094 characters.
2025-03-16 08:36:55,520 - INFO - Completed text chunking for page 1. 3 chunks created.
2025-03-16 08:36:55,582 - INFO - Completed text extraction for dev/page/1806.00663_page_10.pdf. Extracted 1605 characters.
2025-03-16 08:36:55,583 - INFO - Completed text chunking for page 10. 2 chunks created.
2025-03-16 08:36:55,713 - INFO - Completed text extraction for dev/page/1806.00663_page_11.pdf. Extracted 1834 characters.
2025-03-16 08:36:55,714 - INFO - Completed text chunking for page 11. 2 chunks created.
2025-03-16 08:36:56,104 - INFO - Completed text extraction for dev/page/1806.00663_page_12.pdf. Extracted 1956 characters.
2025-03-16 08:36:56,106 - INFO - Completed text chunking for page 12. 2 chunks created.
2025-03-16 08:36:56,154 - INFO - Completed text extraction for dev/page/1806.00663_page_13.pdf. Extracted 2040 characters.
2025-03-16 08:36:56,156 - INFO - Co

Error checking tag for object dev/page/: An error occurred (AccessDenied) when calling the GetObjectTagging operation: User: arn:aws:sts::188494237500:assumed-role/AmazonSageMaker-ExecutionRole-20250122T114147/SageMaker is not authorized to perform: s3:GetObjectTagging on resource: "arn:aws:s3:::sagemaker-us-east-1-188494237500/dev/page/" because no identity-based policy allows the s3:GetObjectTagging action
Error checking tag for object dev/page/1806.00663_page_1.pdf: An error occurred (AccessDenied) when calling the GetObjectTagging operation: User: arn:aws:sts::188494237500:assumed-role/AmazonSageMaker-ExecutionRole-20250122T114147/SageMaker is not authorized to perform: s3:GetObjectTagging on resource: "arn:aws:s3:::sagemaker-us-east-1-188494237500/dev/page/1806.00663_page_1.pdf" because no identity-based policy allows the s3:GetObjectTagging action
Error checking tag for object dev/page/1806.00663_page_10.pdf: An error occurred (AccessDenied) when calling the GetObjectTagging oper

2025-03-16 08:37:26,930 - INFO - Completed listing objects in sagemaker-us-east-1-188494237500/dev/page. 297 files found.
2025-03-16 08:37:26,958 - ERROR - Error extracting text from dev/page/: Cannot read an empty file
2025-03-16 08:37:26,963 - WARNING - Skipping empty text for page None
2025-03-16 08:37:27,041 - INFO - Completed text extraction for dev/page/1806.00663_page_1.pdf. Extracted 3094 characters.
2025-03-16 08:37:27,045 - INFO - Completed text chunking for page 1. 3 chunks created.
2025-03-16 08:37:27,127 - INFO - Completed text extraction for dev/page/1806.00663_page_10.pdf. Extracted 1605 characters.
2025-03-16 08:37:27,128 - INFO - Completed text chunking for page 10. 2 chunks created.


Error checking tag for object dev/page/unwrapping_2011.04041_page_9.pdf: An error occurred (AccessDenied) when calling the GetObjectTagging operation: User: arn:aws:sts::188494237500:assumed-role/AmazonSageMaker-ExecutionRole-20250122T114147/SageMaker is not authorized to perform: s3:GetObjectTagging on resource: "arn:aws:s3:::sagemaker-us-east-1-188494237500/dev/page/unwrapping_2011.04041_page_9.pdf" because no identity-based policy allows the s3:GetObjectTagging action


2025-03-16 08:37:27,203 - INFO - Completed text extraction for dev/page/1806.00663_page_11.pdf. Extracted 1834 characters.
2025-03-16 08:37:27,204 - INFO - Completed text chunking for page 11. 2 chunks created.
2025-03-16 08:37:27,279 - INFO - Completed text extraction for dev/page/1806.00663_page_12.pdf. Extracted 1956 characters.
2025-03-16 08:37:27,280 - INFO - Completed text chunking for page 12. 2 chunks created.
2025-03-16 08:37:27,365 - INFO - Completed text extraction for dev/page/1806.00663_page_13.pdf. Extracted 2040 characters.
2025-03-16 08:37:27,366 - INFO - Completed text chunking for page 13. 2 chunks created.
2025-03-16 08:37:27,412 - INFO - Completed text extraction for dev/page/1806.00663_page_14.pdf. Extracted 616 characters.
2025-03-16 08:37:27,414 - INFO - Completed text chunking for page 14. 1 chunks created.
2025-03-16 08:37:27,448 - INFO - Completed text extraction for dev/page/1806.00663_page_15.pdf. Extracted 569 characters.
2025-03-16 08:37:27,449 - INFO - Co